# 🦜 LangChain Hello World
> **Wire a PromptTemplate, a ChatLLM, and an OutputParser together — the foundation every LangChain application is built on.**

This notebook walks through four short demos, each adding one new layer:
1. Raw LLM call — see what an `AIMessage` looks like
2. `PromptTemplate` — parameterise your prompt
3. Full chain with `StrOutputParser` — get a plain Python string back
4. `.batch()` — process multiple inputs at once

## Setup

Install dependencies and set your OpenAI API key.

```bash
pip install langchain>=0.2.0 langchain-openai>=0.1.0
```

In [ ]:
import os

# Set your OpenAI API key here (or export it in your shell before launching Jupyter)
# os.environ["OPENAI_API_KEY"] = "your-key-here"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialise the LLM — gpt-3.5-turbo is cost-effective for learning
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
print("LLM initialised:", llm.model_name)

## Demo 1 — Raw LLM call

`llm.invoke()` accepts a plain string and returns an `AIMessage` object.
Notice that `.content` holds the actual text — the rest of the object is metadata.
This is what `StrOutputParser` unwraps automatically in later demos.

In [ ]:
response = llm.invoke("Say 'Hello, LangChain!' and explain in one sentence what LangChain is.")

print("Type  :", type(response))
print("Content:", response.content)

## Demo 2 — PromptTemplate

`ChatPromptTemplate` lets you define a prompt once and reuse it with different inputs.
The `{topic}` placeholder is filled at runtime via `.invoke({"topic": "..."})` or `.format_messages()`.

**Why this matters:** keeping prompt logic in a template (not scattered as f-strings) makes prompts
testable, versionable, and easy to swap out without touching application code.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical educator. Answer in 2–3 sentences."),
    ("human", "Explain {topic} to a Python developer who is new to AI."),
])

# Inspect the rendered prompt without calling the LLM
filled = prompt.format_messages(topic="vector embeddings")
print("System:", filled[0].content)
print("Human :", filled[1].content)

In [ ]:
# Pipe prompt into LLM — the | operator creates a Runnable chain
chain = prompt | llm
response = chain.invoke({"topic": "vector embeddings"})
print("AIMessage.content:", response.content)

## Demo 3 — Full chain: PromptTemplate | LLM | StrOutputParser

`StrOutputParser` is the final step that extracts `.content` from the `AIMessage`,
giving you a plain Python `str`. This is the **standard pattern** for chains whose
output feeds into downstream code, other chains, or a user interface.

The pipe `|` operator is part of **LangChain Expression Language (LCEL)** — a declarative
way to compose chains that also enables streaming, batching, and async out of the box.

In [ ]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "Give me one practical use case for {technology} in AWS cloud infrastructure."),
])

# Full chain: prompt → llm → parser
full_chain = prompt2 | llm | StrOutputParser()

# result is a plain Python str
for tech in ["LangChain", "vector databases", "LLM agents"]:
    result = full_chain.invoke({"technology": tech})
    print(f"[{tech}]\n  {result}\n")

## Demo 4 — Batch invocation

`.batch()` accepts a list of input dicts and runs the chain for each one.
This is more efficient than calling `.invoke()` in a Python loop because LangChain
can parallelise the calls under the hood.

In [ ]:
prompt3 = ChatPromptTemplate.from_messages([
    ("system", "You are a technical glossary. Define terms in exactly one sentence."),
    ("human", "Define: {term}"),
])

glossary_chain = prompt3 | llm | StrOutputParser()

terms = [
    {"term": "RAG (Retrieval-Augmented Generation)"},
    {"term": "LangChain Expression Language (LCEL)"},
    {"term": "embedding model"},
]

definitions = glossary_chain.batch(terms)

for item, definition in zip(terms, definitions):
    print(f"{item['term']}:\n  {definition}\n")

## Summary

In this notebook you built four chains using three core LangChain primitives:

| Component | Role |
|-----------|------|
| `ChatOpenAI` | Calls the OpenAI chat API and returns an `AIMessage` |
| `ChatPromptTemplate` | Parameterises prompts so they're reusable and testable |
| `StrOutputParser` | Extracts the plain text from an `AIMessage` |

The `|` pipe operator (LCEL) composes these into a single `Runnable` that supports `.invoke()`, `.batch()`, `.stream()`, and `.ainvoke()` without any extra code.

**What to explore next:**
- **Project 12** — Add `ConversationBufferMemory` to build a chatbot that remembers previous turns
- Try swapping `ChatOpenAI` for `ChatAnthropic` or `ChatBedrock` to see how provider-agnostic LangChain is
- Add a second `ChatPromptTemplate` step to create a two-step chain (generate → critique)